In [11]:
import pandas as pd

df = pd.read_csv("../data/master_dataset_FINAL_FROZEN_READY.csv")


In [18]:
import pandas as pd

df = pd.read_csv("../data/master_dataset_FINAL_FROZEN_READY.csv")


In [19]:
# Normalize bad unicode characters
df["property_age_bucket"] = (
    df["property_age_bucket"]
    .astype(str)
    .str.replace("â€“", "-", regex=False)   # fix broken dash
    .str.replace("–", "-", regex=False)     # normalize en-dash
    .str.replace("—", "-", regex=False)     # normalize em-dash
    .str.strip()
)


In [20]:
def normalize_age_bucket(x):
    if x.startswith("0"):
        return "0-5 years"
    elif x.startswith("5"):
        return "5-10 years"
    elif x.startswith("10"):
        return "10+ years"
    else:
        return "Unknown"

df["property_age_bucket"] = df["property_age_bucket"].apply(normalize_age_bucket)


In [21]:
print(df["property_age_bucket"].value_counts())


property_age_bucket
0-5 years     5790
5-10 years    5127
10+ years     3611
Name: count, dtype: int64


In [22]:
df.to_csv(
    "master_dataset_FINAL_LOCKED_v3.csv",
    index=False,
    encoding="utf-8"
)

print("✅ property_age_bucket FIXED & dataset locked")


✅ property_age_bucket FIXED & dataset locked


In [23]:
df["property_age_bucket"].value_counts()


property_age_bucket
0-5 years     5790
5-10 years    5127
10+ years     3611
Name: count, dtype: int64

In [24]:
import pandas as pd
from datetime import datetime

# ================================
# LOAD FINAL LOCKED DATASET
# ================================
df = pd.read_csv("../data/master_dataset_FINAL_LOCKED_v3.csv")

# ================================
# METADATA POLISH (NO LOGIC CHANGE)
# ================================

# Flood & crime documentation
df["flood_data_level"] = "State"
df["crime_data_level"] = "City"

# Units & scale clarity
df["yield_unit"] = "Percent (%)"
df["investment_score_scale"] = "0–100 (Higher is Better)"

# Dataset governance
df["dataset_version"] = "v2.1"
df["dataset_status"] = "FROZEN_FINAL"
df["last_updated"] = datetime.now().strftime("%Y-%m-%d")
df["data_owner"] = "TY BSc Computer Science – Data Science Project"

# Explicit assumptions (very important for viva)
df["assumption_notes"] = (
    "Flood risk applied at state level due to data availability. "
    "Property age, BHK, and furnishing type inferred using domain assumptions. "
    "Price trend is a proxy, not historical time-series."
)

# ================================
# SAVE POLISHED DATASET
# ================================
output_path = "../data/master_dataset_FINAL_POLISHED.csv"
df.to_csv(output_path, index=False)

print("✅ FINAL METADATA POLISH COMPLETE")
print("📁 Saved as:", output_path)
print("Total columns:", len(df.columns))


✅ FINAL METADATA POLISH COMPLETE
📁 Saved as: ../data/master_dataset_FINAL_POLISHED.csv
Total columns: 37


In [26]:
import pandas as pd

# Load final dataset
df = pd.read_csv("../data/master_dataset_FINAL_POLISHED.csv")

# -------------------------------
# FINAL METADATA POLISH (OPTIONAL)
# -------------------------------

df["dataset_version"] = "v2.0"
df["score_formula_version"] = "RY_70 + SAFETY_20 + FLOOD_10"
df["flood_data_level"] = "State"
df["data_status"] = "FINAL_FROZEN"

# -------------------------------
# SAVE (overwrite or new file)
# -------------------------------
df.to_csv("master_dataset_FINAL_POLISHED.csv", index=False)

print("✅ Final metadata polish applied successfully")


✅ Final metadata polish applied successfully


In [27]:
# Replace existing price_trend_proxy
city_median_price = df.groupby("City_clean")["Price"].median()

df["price_trend_proxy"] = df["City_clean"].map(
    city_median_price.rank(pct=True)
)

df["price_trend_proxy"] = df["price_trend_proxy"].round(3)


In [29]:
rent = pd.read_csv("../data/House_Rent_Dataset.csv")

rent["bhk"] = (
    rent["Size"]
    .astype(str)
    .str.extract(r"(\d+)")
    .astype(float)
)

rent_bhk = (
    rent.groupby(["City", "Area Locality"])["bhk"]
    .median()
    .reset_index()
)


In [30]:
df = df.merge(
    rent_bhk,
    left_on=["City_clean", "Locality"],
    right_on=["City", "Area Locality"],
    how="left"
)


In [34]:
rent = pd.read_csv("../data/House_Rent_Dataset.csv")


In [35]:
rent.columns = rent.columns.str.strip().str.lower()


In [37]:
bhk_map = (
    rent.groupby(["city", "area locality"])["bhk"]
    .median()
    .reset_index()
)


In [38]:
bhk_map.rename(columns={
    "city": "City_clean",
    "area locality": "Locality",
    "bhk": "bhk"
}, inplace=True)


In [39]:
df = df.merge(
    bhk_map,
    on=["City_clean", "Locality"],
    how="left"
)


In [40]:
df["bhk"] = (
    df["bhk"]
    .fillna(df["bhk"].median())
    .round()
    .astype(int)
)

df["bhk"] = df["bhk"].clip(1, 5)


In [41]:
print("BHK distribution:")
print(df["bhk"].value_counts())

print("\nFlood norm zeros:", (df["flood_norm"] == 0).sum())
print("Rent component zeros:", (df["score_rent_component"] == 0).sum())

print("\nInvestment score range:",
      df["investment_score"].min(),
      "→",
      df["investment_score"].max())


BHK distribution:
bhk
2    14155
1      258
3      109
4        5
5        1
Name: count, dtype: int64

Flood norm zeros: 4513
Rent component zeros: 10

Investment score range: 34.5 → 95.1


In [42]:
import pandas as pd

df = pd.read_csv("master_dataset_FINAL_POLISHED.csv")

# See all column names
print(df.columns.tolist())

# Quick preview
df.head()


['property_id', 'City_clean', 'Locality', 'Price', 'Avg_Rent', 'rental_yield_pct', 'yield_category', 'investment_score', 'score_driver', 'score_rent_component', 'score_safety_component', 'score_flood_component', 'property_age_bucket', 'bhk', 'furnishing_type', 'price_trend_proxy', 'Crime_Index', 'Flood_Risk', 'locality_id', 'Name', 'Property Title', 'State', 'Area_sqft', 'ry_prank', 'ry_capped', 'crime_prank', 'safety_prank', 'flood_norm', 'flood_data_level', 'crime_data_level', 'yield_unit', 'investment_score_scale', 'dataset_version', 'dataset_status', 'last_updated', 'data_owner', 'assumption_notes', 'score_formula_version', 'data_status']


,property_id,City_clean,Locality,Price,Avg_Rent,rental_yield_pct,yield_category,investment_score,score_driver,score_rent_component,...,crime_data_level,yield_unit,investment_score_scale,dataset_version,dataset_status,last_updated,data_owner,assumption_notes,score_formula_version,data_status
0,6409ea75-c672-41cb-9c42-154916dfe04d,Chennai,Kanathur Reddikuppam,19900000,14000.0,0.844221,Low,67.8,Rental Yield Driven,6.7,...,City,Percent (%),0–100 (Higher is Better),v2.0,FROZEN_FINAL,2025-12-15,TY BSc Computer Science – Data Science Project,Flood risk applied at state level due to data ...,RY_70 + SAFETY_20 + FLOOD_10,FINAL_FROZEN
1,599c79b0-9ced-437a-9aca-01cdabaf3640,Chennai,Ramanathan Nagar,22500000,13400.0,0.714667,Low,34.5,Rental Yield Driven,5.3,...,City,Percent (%),0–100 (Higher is Better),v2.0,FROZEN_FINAL,2025-12-15,TY BSc Computer Science – Data Science Project,Flood risk applied at state level due to data ...,RY_70 + SAFETY_20 + FLOOD_10,FINAL_FROZEN
2,77077f61-127b-40a4-b751-d2a38649ac28,Chennai,Kasthuribai Nagar,10000000,13400.0,1.608000,Low,34.5,Rental Yield Driven,18.2,...,City,Percent (%),0–100 (Higher is Better),v2.0,FROZEN_FINAL,2025-12-15,TY BSc Computer Science – Data Science Project,Flood risk applied at state level due to data ...,RY_70 + SAFETY_20 + FLOOD_10,FINAL_FROZEN
3,9ac4fb93-ca48-4633-8533-0223036b7a45,Chennai,Naveenilaya,33300000,13400.0,0.482883,Low,34.5,Rental Yield Driven,2.6,...,City,Percent (%),0–100 (Higher is Better),v2.0,FROZEN_FINAL,2025-12-15,TY BSc Computer Science – Data Science Project,Flood risk applied at state level due to data ...,RY_70 + SAFETY_20 + FLOOD_10,FINAL_FROZEN
4,10505f0c-ee90-4653-88c7-91bcf8def8e1,Chennai,Avadi,4800000,8000.0,2.000000,Medium,69.2,Rental Yield Driven,23.4,...,City,Percent (%),0–100 (Higher is Better),v2.0,FROZEN_FINAL,2025-12-15,TY BSc Computer Science – Data Science Project,Flood risk applied at state level due to data ...,RY_70 + SAFETY_20 + FLOOD_10,FINAL_FROZEN


In [88]:
import pandas as pd
import numpy as np

df = pd.read_csv("master_dataset_FINAL_POLISHED.csv")
rent = pd.read_csv("../data/House_Rent_Dataset.csv")


In [90]:
# Clean column names
rent.columns = rent.columns.str.strip()

# Standardize locality column
rent.rename(columns={"Area Locality": "Locality"}, inplace=True)

# Clean strings
rent["City"] = rent["City"].astype(str).str.strip()
rent["Locality"] = rent["Locality"].astype(str).str.strip()


In [91]:
bhk_map = (
    rent
    .groupby(["City", "Locality"], as_index=False)["BHK"]
    .median()
)


In [92]:
df = df.merge(
    bhk_map,
    left_on=["City_clean", "Locality"],
    right_on=["City", "Locality"],
    how="left"
)


In [93]:
df["bhk"] = (
    pd.to_numeric(df["BHK"], errors="coerce")
)

# Fill by city median
df["bhk"] = df["bhk"].fillna(
    df.groupby("City_clean")["bhk"].transform("median")
)

# Global fallback
df["bhk"] = df["bhk"].fillna(df["bhk"].median())

# Final formatting
df["bhk"] = df["bhk"].round().astype(int).clip(1, 5)


In [94]:
df.drop(columns=["City", "BHK"], inplace=True, errors="ignore")


In [95]:
print(df["bhk"].value_counts().sort_index())


bhk
1      258
2    14155
3      109
4        5
5        1
Name: count, dtype: int64


In [96]:
df.to_csv("master_dataset_FINAL_LOCKED_v5.csv", index=False)
print("✅ Final dataset saved as master_dataset_FINAL_LOCKED_v5.csv")


✅ Final dataset saved as master_dataset_FINAL_LOCKED_v5.csv


In [97]:
import pandas as pd
import numpy as np

df = pd.read_csv("master_dataset_FINAL_LOCKED_v5.csv")


In [98]:
# Ensure numeric
df["flood_norm"] = pd.to_numeric(df["flood_norm"], errors="coerce")

# Replace 0 with NaN ONLY where flood data is missing
df.loc[df["flood_data_level"] == "State", "flood_norm"] = (
    df["flood_norm"].replace(0, np.nan)
)

# Fill with median (neutral risk)
df["flood_norm"] = df["flood_norm"].fillna(df["flood_norm"].median())

# Clip safety
df["flood_norm"] = df["flood_norm"].clip(0, 1)


In [99]:
df["score_flood_component"] = (0.1 * df["flood_norm"] * 100).round(2)


In [100]:
df["price_trend_proxy"] = (
    df.groupby("City_clean")["Price"]
      .transform("median")
      .rank(pct=True)
)

df["price_trend_proxy"] = df["price_trend_proxy"].round(3)


In [101]:
print("Zeros check:")
print("flood_norm zeros:", (df["flood_norm"] == 0).sum())
print("price_trend_proxy zeros:", (df["price_trend_proxy"] == 0).sum())
print("score_flood_component zeros:", (df["score_flood_component"] == 0).sum())


Zeros check:
flood_norm zeros: 0
price_trend_proxy zeros: 0
score_flood_component zeros: 0


In [102]:
df.to_csv("master_dataset_FINAL_SUBMISSION.csv", index=False)
print("✅ FINAL DATASET READY FOR PROJECT SUBMISSION")


✅ FINAL DATASET READY FOR PROJECT SUBMISSION


In [105]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("master_dataset_FINAL_SUBMISSION.csv")

print("Before fix:")
print(df["investment_score_scale"].value_counts(dropna=False).head())

# ---------------------------------------
# FIX investment_score_scale (AUTHORITATIVE)
# ---------------------------------------

# Explicitly define the scale (documented assumption)
df["investment_score_scale"] = 100

# ---------------------------------------
# REMOVE UNNECESSARY / DUPLICATE COLUMNS
# ---------------------------------------

cols_to_drop = [
    "dataset_status",   # duplicate of data_status
    "data_owner",       # not needed for submission
    "dataset_version"   # internal tracking only
]

df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

# ---------------------------------------
# FINAL VALIDATION
# ---------------------------------------

print("\nAfter fix:")
print(df["investment_score_scale"].unique())

print("\nRemaining columns:")
print(df.columns.tolist())

# ---------------------------------------
# SAVE FINAL SUBMISSION FILE
# ---------------------------------------

df.to_csv("master_dataset_FINAL_SUBMISSION_CLEAN.csv", index=False)

print("\n✅ FINAL CLEAN DATASET SAVED")
print("📁 File: master_dataset_FINAL_SUBMISSION_CLEAN.csv")


Before fix:
investment_score_scale
0–100 (Higher is Better)    14528
Name: count, dtype: int64

After fix:
[100]

Remaining columns:
['property_id', 'City_clean', 'Locality', 'Price', 'Avg_Rent', 'rental_yield_pct', 'yield_category', 'investment_score', 'score_driver', 'score_rent_component', 'score_safety_component', 'score_flood_component', 'property_age_bucket', 'bhk', 'furnishing_type', 'price_trend_proxy', 'Crime_Index', 'Flood_Risk', 'locality_id', 'Name', 'Property Title', 'State', 'Area_sqft', 'ry_prank', 'ry_capped', 'crime_prank', 'safety_prank', 'flood_norm', 'flood_data_level', 'crime_data_level', 'yield_unit', 'investment_score_scale', 'last_updated', 'assumption_notes', 'score_formula_version', 'data_status']

✅ FINAL CLEAN DATASET SAVED
📁 File: master_dataset_FINAL_SUBMISSION_CLEAN.csv


In [106]:
import pandas as pd
import numpy as np

# Load final dataset
df = pd.read_csv("master_dataset_FINAL_SUBMISSION_CLEAN.csv")

print("Initial invalid price count (< 1 lakh):",
      (df["Price"] < 100000).sum())

# -----------------------------------------
# 1️⃣ Mark invalid prices
# -----------------------------------------
df.loc[df["Price"] < 100000, "Price"] = np.nan

# -----------------------------------------
# 2️⃣ Fill using City median price
# -----------------------------------------
city_price_median = (
    df.groupby("City_clean")["Price"]
      .median()
)

df["Price"] = df["Price"].fillna(
    df["City_clean"].map(city_price_median)
)

# -----------------------------------------
# 3️⃣ Final fallback: global median
# -----------------------------------------
global_median_price = df["Price"].median()
df["Price"] = df["Price"].fillna(global_median_price)

# -----------------------------------------
# 4️⃣ Final sanity checks
# -----------------------------------------
print("\nAfter fix:")
print("Minimum Price:", df["Price"].min())
print("Invalid prices remaining:",
      (df["Price"] < 100000).sum())

# -----------------------------------------
# 5️⃣ Save FINAL LOCKED dataset
# -----------------------------------------
df.to_csv("master_dataset_FINAL_SUBMISSION_LOCKED.csv", index=False)

print("\n✅ Price column fixed")
print("📁 Saved as: master_dataset_FINAL_SUBMISSION_LOCKED.csv")


Initial invalid price count (< 1 lakh): 4

After fix:
Minimum Price: 100000.0
Invalid prices remaining: 0

✅ Price column fixed
📁 Saved as: master_dataset_FINAL_SUBMISSION_LOCKED.csv


In [107]:
df["Price"].describe()


count    1.452800e+04
mean     1.067444e+07
std      1.867258e+07
min      1.000000e+05
25%      3.700000e+06
50%      6.500000e+06
75%      1.140000e+07
max      8.400000e+08
Name: Price, dtype: float64

In [108]:
# Create a validated area column
df["Area_sqft_valid"] = df["Area_sqft"]

# Mark unrealistic small areas as NaN
df.loc[df["Area_sqft"] < 250, "Area_sqft_valid"] = np.nan


In [109]:
# Fill using city median
df["Area_sqft_valid"] = df["Area_sqft_valid"].fillna(
    df.groupby("City_clean")["Area_sqft_valid"].transform("median")
)

# Final fallback
df["Area_sqft_valid"] = df["Area_sqft_valid"].fillna(
    df["Area_sqft_valid"].median()
)


In [110]:
# Fill using city median
df["Area_sqft_valid"] = df["Area_sqft_valid"].fillna(
    df.groupby("City_clean")["Area_sqft_valid"].transform("median")
)

# Final fallback
df["Area_sqft_valid"] = df["Area_sqft_valid"].fillna(
    df["Area_sqft_valid"].median()
)


In [111]:
def area_bucket(x):
    if x < 500:
        return "Small"
    elif x < 1000:
        return "Medium"
    else:
        return "Large"

df["area_category"] = df["Area_sqft_valid"].apply(area_bucket)


In [113]:
import pandas as pd

# Assuming df is already loaded and verified
# Example (only if needed):
# df = pd.read_csv("master_dataset_FINAL_SUBMISSION_CLEAN.csv")

# -------------------------------------------------
# FINAL LOCK & SAVE
# -------------------------------------------------

output_file = "master_dataset_FINAL_SUBMISSION_LOCKED.csv"

df.to_csv(output_file, index=False)

print("✅ FINAL DATASET LOCKED & SAVED")
print("📁 File name:", output_file)
print("📊 Rows:", df.shape[0])
print("📊 Columns:", df.shape[1])


✅ FINAL DATASET LOCKED & SAVED
📁 File name: master_dataset_FINAL_SUBMISSION_LOCKED.csv
📊 Rows: 14528
📊 Columns: 38


In [115]:
import pandas as pd

# Load the locked dataset
df = pd.read_csv("master_dataset_FINAL_SUBMISSION_LOCKED.csv")

# Drop validation/helper column ONLY
cols_to_drop = ["Area_sqft_valid"]

df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

# Final sanity check
print("Remaining columns:")
print(df.columns.tolist())

print("\nArea_sqft stats:")
print(df["Area_sqft"].describe())

# Save FINAL locked dataset
df.to_csv(
    "master_dataset_FINAL_SUBMISSION_LOCKED.csv",
    index=False
)

print("\n✅ FINAL SUBMISSION DATASET LOCKED SUCCESSFULLY")


Remaining columns:
['property_id', 'City_clean', 'Locality', 'Price', 'Avg_Rent', 'rental_yield_pct', 'yield_category', 'investment_score', 'score_driver', 'score_rent_component', 'score_safety_component', 'score_flood_component', 'property_age_bucket', 'bhk', 'furnishing_type', 'price_trend_proxy', 'Crime_Index', 'Flood_Risk', 'locality_id', 'Name', 'Property Title', 'State', 'Area_sqft', 'ry_prank', 'ry_capped', 'crime_prank', 'safety_prank', 'flood_norm', 'flood_data_level', 'crime_data_level', 'yield_unit', 'investment_score_scale', 'last_updated', 'assumption_notes', 'score_formula_version', 'data_status', 'area_category']

Area_sqft stats:
count    14528.000000
mean      1292.054722
std       1209.024856
min         70.000000
25%        650.000000
50%       1000.000000
75%       1430.000000
max      26000.000000
Name: Area_sqft, dtype: float64

✅ FINAL SUBMISSION DATASET LOCKED SUCCESSFULLY


In [119]:
import pandas as pd
import numpy as np

# Load final locked dataset
df = pd.read_csv("master_dataset_FINAL_SUBMISSION_LOCKED.csv")

# Ensure numeric area
df["Area_sqft_valid"] = pd.to_numeric(df["Area_sqft_valid"], errors="coerce")

# Fill missing area using city median (safe assumption)
df["Area_sqft_valid"] = df["Area_sqft_valid"].fillna(
    df.groupby("City_clean")["Area_sqft_valid"].transform("median")
)

# Final fallback (global median)
df["Area_sqft_valid"] = df["Area_sqft_valid"].fillna(
    df["Area_sqft_valid"].median()
)

# ✅ Recruiter-safe area categories
def area_bucket(area):
    if area < 300:
        return "Compact"
    elif area < 600:
        return "Small"
    elif area < 1200:
        return "Medium"
    else:
        return "Large"

df["area_category"] = df["Area_sqft_valid"].apply(area_bucket)

# Final check
print(df["area_category"].value_counts())

# Save FINAL submission dataset
df.to_csv("master_dataset_FINAL_SUBMISSION_LOCKED.csv", index=False)

print("✅ Area cleaned, categorized, and dataset re-locked")


area_category
Medium     6328
Large      5423
Small      2437
Compact     340
Name: count, dtype: int64
✅ Area cleaned, categorized, and dataset re-locked
